    # GNN MAPP: All Modules in One Notebook

    This notebook concatenates all project modules into one Colab-friendly workflow.
    Run cells top-to-bottom to define every module and do a quick end-to-end training smoke test.
    


    ## 1) Colab Setup

    Install runtime dependencies once per session.
    


In [ ]:
%pip -q install "pettingzoo[mpe]" matplotlib


## 2) Shared Imports


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.optim import Adam
from torch.distributions import Categorical
from pettingzoo.mpe import simple_spread_v3


## 3) `utils.py`


In [ ]:
def build_adj(agent_pos, r_comm):
    diff = agent_pos.unsqueeze(1) - agent_pos.unsqueeze(0)
    dist = torch.norm(diff, dim=-1)
    adj = (dist <= r_comm).float()

    deg = adj.sum(dim=1, keepdims=True).clamp(min=1)
    adj = adj / deg
    return adj


def get_agent_pos(env, device):
    base = getattr(env, "unwrapped", env)
    mpe = base

    if not (hasattr(mpe, "world") and hasattr(mpe.world, "agents")):
        mpe = getattr(base, "env", base)
    if not (hasattr(mpe, "world") and hasattr(mpe.world, "agents")):
        mpe = getattr(getattr(base, "env", None), "unwrapped", base)

    if not (hasattr(mpe, "world") and hasattr(mpe.world, "agents")):
        raise AttributeError(
            "Couldn't find MPE world/agents. "
            "This function is intended for PettingZoo MPE simple_spread."
        )

    pos_np = np.stack([agent.state.p_pos for agent in mpe.world.agents], axis=0)
    return torch.as_tensor(pos_np, dtype=torch.float32, device=device)


## 4) `observation.py`


In [ ]:
class ObservationEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ObservationEncoder, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)

        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.orthogonal_(layer.weight, gain=2**0.5)
            nn.init.constant_(layer.bias, 0.0)

    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)

        x = self.fc2(x)
        x = F.relu(x)

        x = self.fc3(x)
        return x


## 5) `graphConv.py`


In [ ]:
class GraphConv(nn.Module):
    def __init__(self, F, G, K):
        super(GraphConv, self).__init__()
        self.F = F
        self.G = G
        self.K = K

        self.weights = nn.ParameterList(
            [nn.Parameter(torch.empty(F, G)) for _ in range(K)]
        )

        for p in self.weights:
            nn.init.xavier_uniform_(p)

    def forward(self, X, S):
        Z = X
        accum = X.new_zeros(*X.shape[:-1], self.G)

        for k in range(self.K):
            accum += torch.matmul(Z, self.weights[k])
            Z = torch.matmul(S, Z)

        return accum


## 6) `action.py`


In [ ]:
class ActionHead(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ActionHead, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self._init_weights()

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.orthogonal_(self.fc2.weight, gain=0.01)
        nn.init.constant_(self.fc2.bias, 0.0)

    def forward(self, G):
        G = self.fc1(G)
        G = F.relu(G)
        G = self.fc2(G)
        return G


## 7) `commPolicy.py`


In [ ]:
class CommPolicy(nn.Module):
    def __init__(self, obs_dim, hidden_dim, action_dim, F, G, K):
        super(CommPolicy, self).__init__()

        self.obsEncoder = ObservationEncoder(obs_dim, hidden_dim, F)
        self.graphConv = GraphConv(F, G, K)
        self.actionHead = ActionHead(G, hidden_dim, action_dim)

    def forward(self, obs, S):
        device = next(self.parameters()).device

        if not torch.is_tensor(obs):
            obs = torch.as_tensor(obs, dtype=torch.float32, device=device)
        else:
            obs = obs.to(device=device, dtype=torch.float32)

        if not torch.is_tensor(S):
            S = torch.as_tensor(S, dtype=torch.float32, device=device)
        else:
            S = S.to(device=device, dtype=torch.float32)

        obs_encode = self.obsEncoder(obs)
        agg_feats = self.graphConv(obs_encode, S)
        logits = self.actionHead(agg_feats)

        return logits

    def get_actions(self, obs, S):
        logits = self.forward(obs, S)
        dist = Categorical(logits=logits)

        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()

        return action, log_prob, entropy

    def evaluate_actions(self, obs, S, actions):
        logits = self.forward(obs, S)
        device = logits.device

        if not torch.is_tensor(actions):
            actions = torch.as_tensor(actions, dtype=torch.long, device=device)
        else:
            actions = actions.to(device=device, dtype=torch.long)

        dist = Categorical(logits=logits)
        log_prob = dist.log_prob(actions)
        entropy = dist.entropy()

        return log_prob, entropy, logits


## 8) `gppoAgent.py`


In [ ]:
class CriticNetwork(nn.Module):
    def __init__(self, obs_dim, hidden_dim, device=None):
        super(CriticNetwork, self).__init__()
        self.fc1 = nn.Linear(obs_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.device = device

        self._init_weights()
        self.to(self.device)

    def _init_weights(self):
        nn.init.orthogonal_(self.fc1.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc2.weight, gain=2**0.5)
        nn.init.orthogonal_(self.fc3.weight, gain=1.0)

        nn.init.constant_(self.fc1.bias, 0.0)
        nn.init.constant_(self.fc2.bias, 0.0)
        nn.init.constant_(self.fc3.bias, 0.0)

    def forward(self, obs):
        if not torch.is_tensor(obs):
            obs = torch.tensor(obs, dtype=torch.float32, device=self.device)
        else:
            obs = obs.to(self.device, dtype=torch.float32)

        x = self.fc1(obs)
        x = F.relu(x)

        x = self.fc2(x)
        x = F.relu(x)

        x = self.fc3(x)
        return x


## 9) `rolloutBuffer.py`


In [ ]:
class GNNRolloutBuffer:
    def __init__(self, gamma, gae_lambda, device):
        self.gamma = gamma
        self.gae_lambda = gae_lambda

        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.adj = []

        self.advantages = None
        self.returns = None

        self.device = device

    def add_timestep(self, obs, actions, rewards, dones, log_probs, values, A):
        self.obs.append(obs)
        self.actions.append(actions)
        self.rewards.append(rewards)
        self.dones.append(dones)
        self.log_probs.append(log_probs)
        self.values.append(values)
        self.adj.append(A)

    def compute_advantages(self, last_values):
        N = len(self.obs[0])
        buffer_size = len(self.obs)
        rewards = torch.stack(self.rewards).to(device=self.device, dtype=torch.float32)
        values = torch.stack(self.values).to(device=self.device, dtype=torch.float32)
        dones = torch.stack(self.dones).to(device=self.device, dtype=torch.float32)

        advantages = torch.zeros((buffer_size, N), dtype=torch.float32, device=self.device)
        last_gae = torch.zeros(len(self.obs[0]), dtype=torch.float32, device=self.device)

        for t in reversed(range(buffer_size)):
            if t == buffer_size - 1:
                if not torch.is_tensor(last_values):
                    next_value = torch.as_tensor(last_values, dtype=torch.float32, device=self.device)
                else:
                    next_value = last_values.to(device=self.device, dtype=torch.float32)
            else:
                next_value = values[t + 1]

            deltas = rewards[t] + self.gamma * (1 - dones[t]) * next_value - values[t]
            advantages[t] = deltas + self.gamma * self.gae_lambda * (1 - dones[t]) * last_gae
            last_gae = advantages[t]

        returns = advantages + values
        self.advantages = advantages
        self.returns = returns

    def get_batches(self, B):
        perm = torch.randperm(len(self.obs), device=self.device)
        batches = perm.split(B)

        obs = torch.stack(self.obs).to(device=self.device, dtype=torch.float32)
        actions = torch.stack(self.actions).to(device=self.device, dtype=torch.long)
        log_probs = torch.stack(self.log_probs).to(device=self.device, dtype=torch.float32)
        adj = torch.stack(self.adj).to(device=self.device, dtype=torch.float32)
        adv_mean = self.advantages.mean()
        adv_std = self.advantages.std() + 1e-8
        advantages = (self.advantages - adv_mean) / adv_std

        for idx in batches:
            m_obs = obs[idx]
            m_actions = actions[idx]
            m_log_probs = log_probs[idx]
            m_advantages = advantages[idx]
            m_returns = self.returns[idx]
            m_adj = adj[idx]

            yield m_obs, m_actions, m_log_probs, m_advantages, m_returns, m_adj

    def clear(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        self.adj = []

        self.advantages = None
        self.returns = None


## 10) `trainer.py`


In [ ]:
class GNNTrainer:
    def __init__(
        self,
        num_agents,
        env,
        obs_dim,
        hidden_dim,
        action_dim,
        F,
        G,
        K,
        lr,
        gamma,
        gae_lambda,
        clip_eps,
        value_coef,
        entropy_coef,
        device,
    ):
        self.device = device
        self.num_agents = num_agents
        self.agent_ids = sorted(env.possible_agents)

        self.clip_eps = clip_eps
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef

        self.comm_policy = CommPolicy(
            obs_dim=obs_dim, hidden_dim=hidden_dim, action_dim=action_dim, F=F, G=G, K=K
        ).to(self.device)
        self.comm_optim = Adam(self.comm_policy.parameters(), lr=lr)

        self.critic = CriticNetwork(obs_dim=obs_dim, hidden_dim=hidden_dim, device=self.device)
        self.critic_optim = Adam(self.critic.parameters(), lr=lr)

        self.buffer = GNNRolloutBuffer(gamma=gamma, gae_lambda=gae_lambda, device=self.device)
        self.env = env
        self._running_episode_return = 0.0
        self.metrics_history = {
            "policy_loss": [],
            "value_loss": [],
            "entropy": [],
            "mean_bellman_error": [],
            "mean_episode_return": [],
            "mean_episode_rewards": [],
        }

        obs, info = self.env.reset()
        self.current_obs = torch.stack([torch.from_numpy(obs[a]) for a in self.agent_ids]).to(
            device=self.device, dtype=torch.float32
        )

    def _safe_mean(self, values):
        if not values:
            return 0.0
        return float(sum(values) / len(values))

    def collect_rollouts(self, num_steps, r_comm=10):
        obs_tensor = self.current_obs
        step_mean_rewards = []
        completed_episode_returns = []

        for _ in range(num_steps):
            agent_pos = get_agent_pos(self.env, self.device)
            S = build_adj(agent_pos, r_comm)

            actions, log_probs, entropy = self.comm_policy.get_actions(obs=obs_tensor, S=S)

            values = self.critic(obs_tensor).detach().squeeze()
            actions_pz = {}
            for i, a_id in enumerate(self.agent_ids):
                actions_pz[a_id] = actions[i].cpu().item()

            next_obs, rewards, dones, truncs, infos = self.env.step(actions_pz)

            rewards_tensor = torch.tensor(
                [rewards[a] for a in self.agent_ids], dtype=torch.float32, device=self.device
            )
            dones_tensor = torch.tensor(
                [dones[a] for a in self.agent_ids], dtype=torch.float32, device=self.device
            )
            self.buffer.add_timestep(
                obs=obs_tensor.detach(),
                actions=actions.detach(),
                rewards=rewards_tensor,
                dones=dones_tensor,
                log_probs=log_probs.detach(),
                values=values,
                A=S.detach(),
            )

            step_mean_rewards.append(rewards_tensor.mean().item())
            self._running_episode_return += rewards_tensor.mean().item()

            if all(dones.values()) or all(truncs.values()):
                completed_episode_returns.append(self._running_episode_return)
                self._running_episode_return = 0.0
                obs, info = self.env.reset()

                obs_tensor = torch.stack([torch.from_numpy(obs[a]) for a in self.agent_ids]).to(
                    device=self.device, dtype=torch.float32
                )
            else:
                obs_tensor = torch.stack([torch.from_numpy(next_obs[a]) for a in self.agent_ids]).to(
                    device=self.device, dtype=torch.float32
                )

            self.current_obs = obs_tensor

        rollout_metrics = {
            "mean_episode_return": self._safe_mean(completed_episode_returns)
            if completed_episode_returns
            else float(self._running_episode_return),
            "mean_episode_rewards": self._safe_mean(step_mean_rewards),
        }
        self.metrics_history["mean_episode_return"].append(rollout_metrics["mean_episode_return"])
        self.metrics_history["mean_episode_rewards"].append(rollout_metrics["mean_episode_rewards"])

        return obs_tensor, rollout_metrics

    def update(self, last_obs, num_epochs=30, B=64):
        with torch.no_grad():
            last_obs_tensor = last_obs.to(device=self.device, dtype=torch.float32)
            last_values = self.critic(last_obs_tensor).squeeze(-1)
        self.buffer.compute_advantages(last_values=last_values)

        policy_losses = []
        entropies = []
        value_losses = []
        bellman_errors = []

        for _ in range(num_epochs):
            for (obs, actions, old_log_probs, advantages, returns, A) in self.buffer.get_batches(B):
                new_lp, entropy, _ = self.comm_policy.evaluate_actions(obs, A, actions)

                ratio = torch.exp(new_lp - old_log_probs)
                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps) * advantages

                policy_loss = -torch.min(surr1, surr2).mean()
                entropy_loss = entropy.mean()
                loss = policy_loss - self.entropy_coef * entropy_loss

                self.comm_optim.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.comm_policy.parameters(), max_norm=0.5)
                self.comm_optim.step()

                policy_losses.append(policy_loss.item())
                entropies.append(entropy_loss.item())

                B_size, N, obs_dim = obs.shape
                flat_obs = obs.reshape(B_size * N, obs_dim)
                flat_returns = returns.reshape(B_size * N)

                pred_values = self.critic(flat_obs).squeeze(-1)
                td_error = flat_returns - pred_values
                value_loss = F.mse_loss(pred_values, flat_returns)
                mean_bellman_error = td_error.abs().mean()

                self.critic_optim.zero_grad()
                value_loss.backward()
                nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
                self.critic_optim.step()

                value_losses.append(value_loss.item())
                bellman_errors.append(mean_bellman_error.item())

        self.buffer.clear()
        update_metrics = {
            "policy_loss": self._safe_mean(policy_losses),
            "value_loss": self._safe_mean(value_losses),
            "entropy": self._safe_mean(entropies),
            "mean_bellman_error": self._safe_mean(bellman_errors),
        }
        self.metrics_history["policy_loss"].append(update_metrics["policy_loss"])
        self.metrics_history["value_loss"].append(update_metrics["value_loss"])
        self.metrics_history["entropy"].append(update_metrics["entropy"])
        self.metrics_history["mean_bellman_error"].append(update_metrics["mean_bellman_error"])

        return update_metrics


## 11) Quick Module Smoke Tests


In [ ]:
torch.manual_seed(0)

obs_dim = 18
hidden_dim = 64
action_dim = 5
F_dim = 64
G_dim = 64
K_hops = 2
N_agents = 3

sample_obs = torch.randn(N_agents, obs_dim)
sample_pos = torch.randn(N_agents, 2)
sample_adj = build_adj(sample_pos, r_comm=1.5)

obs_encoder = ObservationEncoder(obs_dim, hidden_dim, F_dim)
encoded = obs_encoder(sample_obs)
print("ObservationEncoder:", encoded.shape)

gconv = GraphConv(F_dim, G_dim, K_hops)
gconv_out = gconv(encoded, sample_adj)
print("GraphConv:", gconv_out.shape)

action_head = ActionHead(G_dim, hidden_dim, action_dim)
logits = action_head(gconv_out)
print("ActionHead:", logits.shape)

policy = CommPolicy(obs_dim, hidden_dim, action_dim, F_dim, G_dim, K_hops)
acts, logp, ent = policy.get_actions(sample_obs, sample_adj)
print("CommPolicy actions/logp/entropy:", acts.shape, logp.shape, ent.shape)

critic = CriticNetwork(obs_dim, hidden_dim, device="cpu")
values = critic(sample_obs)
print("CriticNetwork:", values.shape)


    ## 12) End-to-End Training Demo (`mapp.py` style)

    This is a short smoke run tuned for notebook execution time.
    Increase `TOTAL_TIMESTEPS`, `ROLLOUT_LENGTH`, and `NUM_EPOCHS` for full training.
    


In [ ]:
NUM_AGENTS = 3
MAX_CYCLES = 100

env = simple_spread_v3.parallel_env(N=NUM_AGENTS, max_cycles=MAX_CYCLES)
obs, info = env.reset()
obs_dim = obs["agent_0"].shape[0]
action_dim = 5

F_dim = 64
G_dim = 64
K_hops = 2
hidden_dim = 64

lr = 3e-4
gamma = 0.99
gae_lambda = 0.85
clip_eps = 0.2
value_coef = 0.5
entropy_coef = 0.01

TOTAL_TIMESTEPS = 2048
ROLLOUT_LENGTH = 256
BATCH_SIZE = 64
NUM_EPOCHS = 2
R_COMM = 1.0

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"

print(f"Using device: {device}")

trainer = GNNTrainer(
    num_agents=NUM_AGENTS,
    env=env,
    obs_dim=obs_dim,
    hidden_dim=hidden_dim,
    action_dim=action_dim,
    F=F_dim,
    G=G_dim,
    K=K_hops,
    lr=lr,
    gamma=gamma,
    gae_lambda=gae_lambda,
    clip_eps=clip_eps,
    value_coef=value_coef,
    entropy_coef=entropy_coef,
    device=device,
)

num_iterations = max(1, TOTAL_TIMESTEPS // ROLLOUT_LENGTH)

for iteration in range(num_iterations):
    last_obs, rollout_metrics = trainer.collect_rollouts(num_steps=ROLLOUT_LENGTH, r_comm=R_COMM)
    update_metrics = trainer.update(last_obs, num_epochs=NUM_EPOCHS, B=BATCH_SIZE)

    print(
        f"Iteration {iteration + 1}/{num_iterations} | "
        f"policy_loss={update_metrics['policy_loss']:.4f} | "
        f"value_loss={update_metrics['value_loss']:.4f} | "
        f"entropy={update_metrics['entropy']:.4f} | "
        f"mean_bellman_error={update_metrics['mean_bellman_error']:.4f} | "
        f"mean_episode_return={rollout_metrics['mean_episode_return']:.4f} | "
        f"mean_episode_rewards={rollout_metrics['mean_episode_rewards']:.4f}"
    )

metrics_to_plot = [
    "policy_loss",
    "value_loss",
    "entropy",
    "mean_bellman_error",
    "mean_episode_return",
    "mean_episode_rewards",
]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, metric_name in enumerate(metrics_to_plot):
    values = trainer.metrics_history.get(metric_name, [])
    x = range(1, len(values) + 1)
    axes[i].plot(x, values, linewidth=1.8)
    axes[i].set_title(metric_name)
    axes[i].set_xlabel("Iteration")
    axes[i].set_ylabel(metric_name)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/training_metrics.png", dpi=180)
plt.show()
print("Saved metrics plot to outputs/training_metrics.png")
